# DocAgent: week 1 on Kaggle

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **On**.

**Every session:** run the cells from top to bottom, one at a time. Each step skips itself if its data is already there.

**Keep your data between sessions:** after a successful run, click **Save Version → Save & Run All**. In the next session, add it with **Add Input → Your Work → docagent-week1**, and the setup cell restores everything in about a minute.

**Schneider manual:** keep your `manuals` dataset attached (Add Input → Your Datasets) so step M3 can find it.

## Setup (run first, and again after any restart)

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "DocAgent"
os.chdir(WORK)

def _copy_missing(src, dst):
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
    return dst

# Repair: an old "git clone + %cd" cell may have nested copies (DocAgent/DocAgent/...).
# Keep the innermost repo, merge any data found in the outer copies, and move it to the top.
inner = REPO
while (inner / "DocAgent" / "docagent").is_dir():
    inner = inner / "DocAgent"
if inner != REPO:
    print(f"Repairing nested folders: keeping {inner}")
    level = inner.parent
    while level != WORK:
        for part in ("data", "results"):
            if (level / part).is_dir():
                shutil.copytree(level / part, inner / part, dirs_exist_ok=True, copy_function=_copy_missing)
        level = level.parent
    shutil.move(str(inner), str(WORK / "_DocAgent_fixed"))
    shutil.rmtree(REPO)
    shutil.move(str(WORK / "_DocAgent_fixed"), str(REPO))

if not REPO.is_dir():
    subprocess.run(["git", "clone", "-q", "https://github.com/ademghorbel/DocAgent.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull", "-q"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Working in", os.getcwd())

def sh(cmd):
    """Run a shell command with live output; stop the cell if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"Command failed (exit {code}): {cmd}")

def have(path):
    return Path(path).exists()

sh("pip install -q -r requirements.txt")
sh("python scripts/restore_data.py")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

## Status: what is already prepared

In [ ]:
import json
def status():
    for name in ("mmlongbench", "vidore_industrial", "manuals"):
        qf = Path(f"data/questions/{name}.jsonl") if name != "manuals" else Path("data/questions/own.jsonl")
        pf = Path(f"data/pages/{name}/pages.jsonl")
        nq = sum(1 for _ in open(qf)) if qf.exists() else 0
        npg = sum(1 for _ in open(pf)) if pf.exists() else 0
        print(f"{name:18s} questions: {nq:4d}   pages: {npg:5d}")
status()

## M1: tests (CPU only, a few seconds)

In [ ]:
sh("python -m pytest -q")

## M2: benchmarks (skipped if already prepared)
MMLongBench: ~5 min. ViDoRe industrial: 15–20 min (5,244 page images).

In [ ]:
if have("data/questions/mmlongbench.jsonl") and have("data/pages/mmlongbench/pages.jsonl"):
    print("MMLongBench already prepared, skipping.")
else:
    sh("python scripts/prepare_mmlongbench.py --n 100 --max-docs 15")

In [ ]:
if have("data/questions/vidore_industrial.jsonl") and have("data/pages/vidore_industrial/pages.jsonl"):
    print("ViDoRe already prepared, skipping.")
else:
    sh("python scripts/prepare_vidore.py --n 60")

## M3: own manuals (skipped if already prepared)
Downloads the manuals in `data/manuals.csv` and adds the Schneider PDF from your attached `manuals` dataset. About 15 minutes.

In [ ]:
if have("data/pages/manuals/pages.jsonl"):
    print("Manuals already prepared, skipping.")
else:
    sh("python scripts/download_manuals.py --from /kaggle/input")
status()

## Data fixes (safe to rerun, each part runs once)
1. Removes MMLongBench evidence pages numbered 0 (an annotation error in the benchmark).
2. OCRs pages that are only images, so the text baseline can search them. About 5 minutes the first time.

In [ ]:
qf = Path("data/questions/mmlongbench.jsonl")
if qf.exists() and any(e["page"] < 1 for l in open(qf) for e in json.loads(l)["evidence"]):
    sh("python scripts/prepare_mmlongbench.py --n 100 --max-docs 15")   # PDFs and pages are reused
else:
    print("MMLongBench evidence pages OK.")

sh("apt-get install -y -qq tesseract-ocr > /dev/null && pip install -q pytesseract")
sh("python scripts/ocr_scanned.py")

## Look at one ViDoRe question and its evidence page (optional)

In [ ]:
from IPython.display import Image, display
from docagent.config import CFG
from docagent.data import PageStore, load_questions

qs = load_questions("data/questions/vidore_industrial.jsonl")
store = PageStore(CFG.pages_dir, "vidore_industrial")
q = qs[0]   # change the index to look at other questions
print("Q:", q.question, "\nTags:", q.tags, "\nAnswer:", q.answer[:400])
page = store.get(q.evidence[0]["doc_id"], q.evidence[0]["page"])
if page is None:
    print("Evidence page not found: is ViDoRe prepared? Run the status cell.")
else:
    print(f"\nEvidence: {page.doc_id} page {page.page}\n--- page text ---\n{page.text[:800]}")
    display(Image(str(store.image_path(page)), width=550))

## M4: smoke test, 5 questions per set
The first run downloads the VLM (~16 GB, about 2 minutes on Kaggle), then answers 10 questions.

In [ ]:
shutil.rmtree("results/smoke", ignore_errors=True)   # always a fresh smoke test
sh("python eval/run_eval.py --questions data/questions/mmlongbench.jsonl data/questions/vidore_industrial.jsonl "
   "--systems B1 B2 --limit 5 --run smoke")

In [ ]:
# Raw replies: check the model follows "Answer: ... / Pages: ..."
for line in open("results/smoke/predictions.jsonl"):
    r = json.loads(line)
    print(r["system"], r["qid"], "score", r.get("score"), "|", (r["raw"] or r["skipped"])[:300].replace("\n", " / "))

In [ ]:
# Diagnostic: gold answer and evidence pages vs what B1 retrieved
qs_all = {q.qid: q for f in ["data/questions/mmlongbench.jsonl", "data/questions/vidore_industrial.jsonl"]
          for q in load_questions(f)}
for line in open("results/smoke/predictions.jsonl"):
    r = json.loads(line)
    if r["system"] != "B1_text_rag":
        continue
    q = qs_all[r["qid"]]
    print(f"--- {q.qid} {q.tags}")
    print("Q:         ", q.question[:150])
    print("Gold:      ", q.answer[:150])
    print("Gold pages:", sorted(q.evidence_keys))
    print("Retrieved: ", [tuple(x) for x in r["retrieved"]])
    print("Pred:      ", r["prediction"][:150])

## M5: check your own questions
Write `data/questions/own.jsonl` first (copy the format from `own.example.jsonl`), push it, then rerun the setup cell.

In [ ]:
if have("data/questions/own.jsonl"):
    sh("python scripts/check_questions.py data/questions/own.jsonl")
else:
    print("No own.jsonl yet: write your questions first (planned for Friday).")

## M6: full run
Set `RUN_FULL = True` when you are ready. It takes 30–60 minutes. If the session stops, rerun: finished questions are skipped.

Optional: add a Kaggle secret `WANDB_API_KEY` to log the run to Weights & Biases.

In [ ]:
RUN_FULL = False

if not RUN_FULL:
    print("Full run is off. Set RUN_FULL = True to start it.")
else:
    files = [f for f in ["data/questions/mmlongbench.jsonl", "data/questions/vidore_industrial.jsonl",
                         "data/questions/own.jsonl"] if have(f)]
    use_wandb = ""
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        use_wandb = "--wandb"
    except Exception:
        pass
    print("Questions:", files, "| W&B:", "on" if use_wandb else "off")
    sh(f"python eval/run_eval.py --questions {' '.join(files)} --systems B1 B2 --judge llm --run week1 {use_wandb}")

## M7: results and error analysis

In [ ]:
from IPython.display import Markdown
if have("results/week1/summary.md"):
    display(Markdown(open("results/week1/summary.md").read()))
    sh("python eval/show_errors.py --run week1 --questions data/questions/*.jsonl -n 10")
else:
    print("No full results yet (run M6 first).")

## Before you leave
Click **Save Version → Save & Run All**. Everything above is safe to rerun: prepared data is skipped and the full run stays off unless `RUN_FULL = True`.